# Notebook 21 — RECOVER original official test without retraining

The preceding Notebook 20 already completed **58 frozen training rounds** and **one official-test inference**. It then failed while accessing a misspelled **published-paper comparator key**: `macro_f1` instead of the pre-registered `macro_f1_reported`.

**This notebook never trains a model and never opens the original UNSW training or test CSV.** It repairs only the missing comparison and final audit/backup of EXISTING immutable files from the original first evaluation.

### Important requirements
- Work within the **same still-open Kaggle runtime** that executed Notebook 20, or copy that run's ORIGINAL output directory `/kaggle/working/research_outputs/WINDOW-OFFICIAL-001-S42` from the existing Kaggle Version Outputs.
- **Do not Run All on Notebook 20 again**: the official test has already been used.
- Internet ON and the existing GitHub `pushRare` secret enabled for private checkpoint/prediction backups.
- **If any original model/prediction files are missing**, do not retrain or recompute the official test. The recovery script will stop and list which original files must first be restored from Kaggle Outputs.
- The original checkpoint SHA256 from the user's run was `8d68c55d74f4ac09f5472ae860fd05b508f6ea4f5c3902d9c92c81286b97c353`, archived BEFORE test at commit `a8ec5f4dbd96c80179fc12d29889692f18976ef8`.
- When completed, it privately uploads original genuine per-record predictions and commits only numeric metrics/confusions to GitHub. This does not alter hyperparameters, predictions, labels, model, or raw official test dataset.

**Success marker:** `OFFICIAL-001 RECOVERED RESULTS AUDITED AND PRIVATELY ARCHIVED`.

In [ ]:
# SINGLE CELL: verify preexisting original outputs, repair typo, archive. NO MODEL TRAINING.
import os,sys,json,subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

OUT=Path("/kaggle/working/research_outputs/WINDOW-OFFICIAL-001-S42")
REPO=Path("/kaggle/working/rare-attack-ids-official-001")
NEEDED=("final_model_PRIVATE.txt","PRETEST_FROZEN_CHECKPOINT.json",
        "official_metrics.json","official_test_predictions_PRIVATE.npz",
        "official_confusion.csv","official_per_class.csv","config_effective.json")
missing=[p for p in NEEDED if not (OUT/p).is_file()]
if missing:
    raise FileNotFoundError(
        "STOP — the original official test has already been used. "
        "DO NOT rerun Notebook 20. Restore these EXACT original files from the "
        "existing Kaggle Version Outputs before proceeding: "+", ".join(missing))
original=json.loads((OUT/"official_metrics.json").read_text())
freeze=json.loads((OUT/"PRETEST_FROZEN_CHECKPOINT.json").read_text())
assert freeze["model_checkpoint_sha256"]==(
    "8d68c55d74f4ac09f5472ae860fd05b508f6ea4f5c3902d9c92c81286b97c353"
),"Pre-test fitted model differs from the originally published GitHub freeze"
print("ORIGINAL OFFICIAL TEST ACCURACY: {:.4%}".format(original["accuracy"]),flush=True)
print("ORIGINAL OFFICIAL TEST MACRO-F1: {:.4%}".format(original["macro_f1"]),flush=True)
print("ORIGINAL OFFICIAL TEST WEIGHTED-F1: {:.4%}".format(original["weighted_f1"]),flush=True)
print("TEST ROWS:",original["n"],flush=True)
print("No training or test access is required; only original saved outputs will be used.",flush=True)
assert (REPO/".git").is_dir(), "Original Kaggle GitHub checkout is missing; restore or clone it, but do not run Notebook 20 again"
subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
secret=UserSecretsClient().get_secret("pushRare")
assert secret,"Enable the existing pushRare Kaggle secret for private GitHub archiving"
env=os.environ.copy()
env["GITHUB_TOKEN"]=secret
env["PYTHONPATH"]=str(REPO/"src")
subprocess.run([sys.executable,str(REPO/"scripts/recover_official_001.py"),
                "--output",str(OUT),"--archive"],
               cwd=str(REPO),env=env,check=True)
print("DONE: OFFICIAL SCORE RECOVERED FROM THE ONE AND ONLY ORIGINAL TEST",flush=True)
